# 📄 고영향 인공지능 사업자 책무 가이드라인 전처리

**문서 구조**
- 총 111페이지
- 책무 항목 번호: `1-1-1`, `1-1-2` 형태
- 항목 내부: **목표 / 설명 / 참고** 3단 구조
- 5개 책무: 위험관리 / 설명 / 이용자보호 / 관리감독 / 문서보관
- 부록: 자가점검, 채용분야 예시

**전처리 전략**
1. 전체 텍스트 추출 + 클리닝
2. 책무 조치사항(3.1~3.5) 영역 분리
3. `1-1-1` 번호 기준 개별 청크 + 목표/설명/참고 파싱
4. FAQ 섹션 별도 추출
5. JSONL 저장

In [ ]:
!pip install pdfplumber -q

In [ ]:
import pdfplumber, re, json
from collections import Counter

PDF_PATH = "고영향_인공지능_사업자_책무_가이드라인.pdf"   # ← 경로 확인
OUTPUT_PATH = "책무가이드라인_chunks.jsonl"


## 1단계: 추출 + 클리닝

In [ ]:
def extract_and_clean(pdf_path):
    texts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t and t.strip():
                texts.append(t.strip())
    full = "\n\n".join(texts)
    full = re.sub(r'고영향 인공지능사업자\s*책무 가이드라인\s*', '', full)
    full = re.sub(r'^\s*\d+\s*$', '', full, flags=re.MULTILINE)
    full = full.replace('Ÿ', '•').replace('ㅅ', '')
    full = re.sub(r'[ \t]{2,}', ' ', full)
    full = re.sub(r'\n{3,}', '\n\n', full)
    print(f"유효 페이지: {len(texts)}, 총 {len(full)}자")
    return full

full = extract_and_clean(PDF_PATH)
print(full[:400])


## 2단계: 책무 조치사항 영역 분리

In [ ]:
CHAPTER_PATTERNS = {
    "위험관리": r'3\.1[\s]+위험관리방안',
    "설명방안": r'3\.2[\s]+기술적으로 가능한 범위',
    "이용자보호": r'3\.3[\s]+이용자 보호',
    "관리감독": r'3\.4[\s]+사람의 관리',
    "문서보관": r'3\.5[\s]+안전성',
    "부록": r'부록|자가점검 항목',
}

positions = {}
for key, pat in CHAPTER_PATTERNS.items():
    m = re.search(pat, full)
    if m:
        positions[key] = m.start()
        print(f"  {key}: 위치 {m.start()}")
    else:
        print(f"  {key}: 미발견 (패턴 확인 필요)")

책무_start = positions.get("위험관리", 0)
책무_end = positions.get("부록", len(full))
책무_text = full[책무_start:책무_end]
print(f"\n책무 조치사항 영역: {len(책무_text)}자")


## 3단계: 1-1-1 항목 추출 + 목표/설명/참고 파싱

In [ ]:
SECTION_MAP = {
    "1": "위험관리방안 수립·운영",
    "2": "설명방안 수립·시행",
    "3": "이용자 보호 방안 수립·운영",
    "4": "사람의 관리·감독",
    "5": "문서 작성과 보관",
}

ITEM_PAT = re.compile(
    r'(\d+-\d+-\d+)[.\s]+([^\n]{5,80})\n(.*?)(?=\d+-\d+-\d+[.\s]|$)',
    re.DOTALL
)

def parse_fields(body):
    def find(label, after=None):
        pat = rf'{label}\s+(.+?)(?=목표|설명|참고|$)'
        after_labels = '|'.join(l for l in ['목표','설명','참고'] if l != label)
        pat = rf'{label}\s+(.+?)(?={after_labels}|$)'
        m = re.search(pat, body, re.DOTALL)
        return re.sub(r'\s+', ' ', m.group(1)).strip() if m else ""
    return {"목표": find("목표"), "설명": find("설명"), "참고": find("참고")}

def extract_obligations(text):
    chunks = []
    for m in ITEM_PAT.finditer(text):
        item_num = m.group(1)
        title = m.group(2).strip()
        body = m.group(3).strip()
        major = item_num.split("-")[0]
        fields = parse_fields(body)
        
        is_dev = "개발사업자" in body
        is_user = "이용사업자" in body
        target = ("개발·이용사업자" if is_dev and is_user
                  else "개발사업자" if is_dev
                  else "이용사업자" if is_user else "공통")
        
        content = f"[{SECTION_MAP.get(major,'기타')}] {item_num} {title}\n"
        if fields["목표"]: content += f"목표: {fields['목표']}\n"
        if fields["설명"]: content += f"설명: {fields['설명'][:500]}"
        
        chunks.append({
            "chunk_id": f"책무_{item_num}",
            "항목번호": item_num,
            "대분류": SECTION_MAP.get(major, "기타"),
            "제목": title,
            "목표": fields["목표"],
            "설명": fields["설명"][:800],
            "참고": fields["참고"][:300],
            "적용대상": target,
            "content": content,
            "출처": "고영향 인공지능 사업자 책무 가이드라인",
            "청크유형": "책무항목"
        })
    print(f"책무 항목: {len(chunks)}개 추출")
    dist = Counter(c["대분류"] for c in chunks)
    for k, v in dist.items():
        print(f"  {k}: {v}개")
    return chunks

obligation_chunks = extract_obligations(책무_text)
if obligation_chunks:
    s = obligation_chunks[0]
    print(f"\n[샘플] {s['chunk_id']} {s['제목']}")
    print(f"목표: {s['목표'][:80]}")
    print(f"설명: {s['설명'][:120]}...")


## 4단계: FAQ 추출

In [ ]:
def extract_faq(text):
    chunks = []
    pat = re.compile(r'Q\s+(.+?)\s+A\s+(.+?)(?=Q\s|$)', re.DOTALL)
    for i, m in enumerate(pat.finditer(text)):
        q = re.sub(r'\s+', ' ', m.group(1)).strip()
        a = re.sub(r'\s+', ' ', m.group(2)).strip()
        if len(q) < 10 or len(a) < 20:
            continue
        chunks.append({
            "chunk_id": f"FAQ_{i+1:02d}",
            "질문": q,
            "답변": a[:800],
            "content": f"Q: {q}\nA: {a[:600]}",
            "출처": "고영향 인공지능 사업자 책무 가이드라인",
            "청크유형": "FAQ"
        })
    print(f"FAQ: {len(chunks)}개 추출")
    return chunks

faq_chunks = extract_faq(full)


## 5단계: 저장

In [ ]:
all_chunks = obligation_chunks + faq_chunks
print(f"총 청크: {len(all_chunks)}개 (책무 {len(obligation_chunks)} + FAQ {len(faq_chunks)})")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"저장 완료 → {OUTPUT_PATH}")
lengths = [len(c["content"]) for c in all_chunks]
print(f"평균 청크 길이: {sum(lengths)//len(lengths)}자")
